# DNB — Batch Offline Processing

Processes a set of `.npz` recordings (ns6-converted or synthetic). For each file:

1. Runs the full pipeline (downsampler → wavelet → detector → inhibitor → trigger)
2. Saves a detection log (`.csv`) with sample indices at the **original hardware rate**
3. Produces a 3-panel report figure (stim-triggered average, phase accuracy, fired vs blocked)

The detection log is for validating detections by eye against the raw recording.

### .npz format

Use `ns6_to_npz.py` to convert Blackrock `.ns6` files. The resulting `.npz` contains:
- `data`: `(n_samples, n_channels)` int16
- `fs`: sample rate (Hz)
- `scale_factors`: `(n_channels,)` — multiply int16 to get µV
- `electrode_ids`, `labels`: channel metadata

`FileSource` reads `fs` and `scale_factors` from the file automatically.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from math import pi
import csv
import logging

from dnb import Pipeline, FileSource, PipelineConfig, EventType
from dnb.modules import (
    WaveletConvolution, TargetWaveDetector, AmplitudeMonitor, StimTrigger,
    Downsampler,
)
from scipy.signal import butter, sosfilt

import dnb
print(f'DNB v{dnb.__version__}')

logging.basicConfig(level=logging.WARNING)

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

In [ ]:
# ── Files to process ──────────────────────────────────────────────────
# List of .npz paths. Sample rate is read from each file automatically.

FILES = [
    Path('D:/DanH/20190122-065346-004.npz'),
    # Path('D:/DanH/another_recording.npz'),
]

OUTPUT_DIR = Path('../output/batch')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Channel ────────────────────────────────────────────────────────────
CHANNEL_ID = 0               # which channel to extract

# ── Pipeline parameters ────────────────────────────────────────────────
ANALYSIS_RATE = 500.0
CHUNK_DURATION = 0.05
BUFFER_DURATION = 10.0

# Wavelet
FREQ_MIN = 0.5
FREQ_MAX = 4.0
N_FREQS = 20
N_CYCLES_BASE = 1.0

# Detector
DETECTION_PHASE = pi          # trough
PHASE_TOLERANCE = 0.05
Z_SCORE_THRESHOLD = 1.5
WARMUP_CHUNKS = 20

# Inhibitor
IED_FREQ_RANGE = (60.0, 120.0)
IED_ADAPTIVE_N_STD = 5.0
IED_WARMUP = 20

# Trigger
STIM_PHASE = 0.0              # peak
N_PULSES = 1
BACKOFF_S = 0.0
INHIBITION_COOLDOWN_S = 2.5

# ── Validate ───────────────────────────────────────────────────────────
print(f'Output: {OUTPUT_DIR.resolve()}')
print(f'\nFiles ({len(FILES)}):')
for path in FILES:
    path = Path(path)
    if path.exists():
        npz = np.load(str(path), allow_pickle=True)
        keys = list(npz.keys())
        fs = float(npz.get('fs', npz.get('sample_rate', 0)))
        data_key = 'data' if 'data' in keys else 'continuous' if 'continuous' in keys else '?'
        shape = npz[data_key].shape if data_key in keys else '?'
        print(f'  \u2713  {path.name}  fs={fs:.0f}  {data_key}={shape}')
    else:
        print(f'  \u2717  {path.name}  NOT FOUND')

In [ ]:
def get_file_info(npz_path):
    """Read sample rate from a .npz file."""
    npz = np.load(str(npz_path), allow_pickle=True)
    if 'fs' in npz:
        return float(npz['fs'])
    elif 'sample_rate' in npz:
        return float(npz['sample_rate'])
    else:
        raise KeyError(f"No sample rate key in {npz_path.name}. Keys: {list(npz.keys())}")


def make_pipeline(npz_path, hw_rate):
    """Build pipeline WITH IED inhibition."""
    return Pipeline(
        source=FileSource(str(npz_path)),
        modules=[
            Downsampler(target_rate=ANALYSIS_RATE),
            WaveletConvolution(
                freq_min=FREQ_MIN, freq_max=FREQ_MAX,
                n_freqs=N_FREQS, n_cycles_base=N_CYCLES_BASE,
            ),
            TargetWaveDetector(
                id='slow_wave', freq_range=(FREQ_MIN, FREQ_MAX),
                detection_phase=DETECTION_PHASE,
                phase_tolerance=PHASE_TOLERANCE,
                z_score_threshold=Z_SCORE_THRESHOLD,
                warmup_chunks=WARMUP_CHUNKS,
            ),
            AmplitudeMonitor(
                id='ied_monitor', freq_range=IED_FREQ_RANGE,
                adaptive_n_std=IED_ADAPTIVE_N_STD,
                warmup_chunks=IED_WARMUP,
            ),
            StimTrigger(
                activation_detector_id='slow_wave',
                inhibition_detector_id='ied_monitor',
                detection_phase=DETECTION_PHASE,
                stim_phase=STIM_PHASE,
                n_pulses=N_PULSES,
                backoff_s=BACKOFF_S,
                inhibition_cooldown_s=INHIBITION_COOLDOWN_S,
            ),
        ],
        config=PipelineConfig(
            sample_rate=hw_rate,
            channel_id=CHANNEL_ID,
            buffer_duration=BUFFER_DURATION,
            chunk_duration=CHUNK_DURATION,
        ),
    )


def make_pipeline_no_inh(npz_path, hw_rate):
    """Build pipeline WITHOUT IED inhibition."""
    return Pipeline(
        source=FileSource(str(npz_path)),
        modules=[
            Downsampler(target_rate=ANALYSIS_RATE),
            WaveletConvolution(
                freq_min=FREQ_MIN, freq_max=FREQ_MAX,
                n_freqs=N_FREQS, n_cycles_base=N_CYCLES_BASE,
            ),
            TargetWaveDetector(
                id='slow_wave', freq_range=(FREQ_MIN, FREQ_MAX),
                detection_phase=DETECTION_PHASE,
                phase_tolerance=PHASE_TOLERANCE,
                z_score_threshold=Z_SCORE_THRESHOLD,
                warmup_chunks=WARMUP_CHUNKS,
            ),
            StimTrigger(
                activation_detector_id='slow_wave',
                inhibition_detector_id=None,
                detection_phase=DETECTION_PHASE,
                stim_phase=STIM_PHASE,
                n_pulses=N_PULSES,
                backoff_s=BACKOFF_S,
                inhibition_cooldown_s=INHIBITION_COOLDOWN_S,
            ),
        ],
        config=PipelineConfig(
            sample_rate=hw_rate,
            channel_id=CHANNEL_ID,
            buffer_duration=BUFFER_DURATION,
            chunk_duration=CHUNK_DURATION,
        ),
    )


def load_signal_ds(npz_path, hw_rate):
    """Load .npz -> 1D signal at analysis rate for plotting."""
    npz = np.load(str(npz_path), allow_pickle=True)
    if 'data' in npz:
        raw = npz['data']
        if raw.ndim == 2:
            ch = min(CHANNEL_ID, raw.shape[1] - 1)
            sig = raw[:, ch].astype(np.float64)
        else:
            sig = raw.astype(np.float64)
        if 'scale_factors' in npz:
            sf = npz['scale_factors']
            ch_sf = min(CHANNEL_ID, len(sf) - 1)
            sig *= float(sf[ch_sf])
    elif 'continuous' in npz:
        raw = npz['continuous'].astype(np.float64)
        if raw.ndim == 2:
            ch = min(CHANNEL_ID, raw.shape[0] - 1)
            sig = raw[ch]
        else:
            sig = raw
    else:
        raise KeyError(f"No data key in {npz_path}")

    ds_factor = max(1, int(round(hw_rate / ANALYSIS_RATE)))
    return sig[::ds_factor]


def timestamp_to_hw_index(timestamp, hw_rate):
    """Convert pipeline timestamp (seconds) to hardware sample index."""
    return int(round(timestamp * hw_rate))


def save_detection_log(csv_path, events, hw_rate):
    """Save detection log as CSV with hardware-rate sample indices."""
    with open(csv_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([
            'event_type', 'timestamp_s', 'sample_index',
            'frequency_hz', 'amplitude', 'z_score',
            'pulse_index', 'detection_time_s',
        ])
        for e in events:
            idx = timestamp_to_hw_index(e.timestamp, hw_rate)
            if e.event_type == EventType.SLOW_WAVE:
                writer.writerow([
                    'SLOW_WAVE', f'{e.timestamp:.4f}', idx,
                    f'{e.metadata.get("frequency", 0):.3f}',
                    f'{e.metadata.get("amplitude", 0):.1f}',
                    f'{e.metadata.get("z_score", 0):.2f}',
                    '', '',
                ])
            elif e.event_type == EventType.STIM:
                det_t = e.metadata.get('detection_time', 0)
                writer.writerow([
                    'STIM', f'{e.timestamp:.4f}', idx,
                    f'{e.metadata.get("frequency", 0):.3f}',
                    '', '',
                    e.metadata.get('pulse_index', ''),
                    f'{det_t:.4f}',
                ])
    return csv_path


print('Helpers ready.')

In [ ]:
def make_report_figure(sig_ds, stims_with, stims_without, all_events, filename, save_path=None):
    """3-panel report figure for one recording.

    stims_with:    stims from pipeline WITH inhibition
    stims_without: stims from pipeline WITHOUT inhibition
    """
    win_s = 2.0
    win_samples = int(win_s * ANALYSIS_RATE)
    t_win = np.arange(-win_samples, win_samples) / ANALYSIS_RATE

    # Filter to SO band
    nyq = ANALYSIS_RATE / 2.0
    sos = butter(4, [FREQ_MIN / nyq, FREQ_MAX / nyq], btype='band', output='sos')
    sig_so = sosfilt(sos, sig_ds)

    # Extract windows from filtered signal
    stim_windows = []
    for e in stims_with:
        centre = int(e.timestamp * ANALYSIS_RATE)
        lo, hi = centre - win_samples, centre + win_samples
        if lo >= 0 and hi < len(sig_so):
            stim_windows.append(sig_so[lo:hi])

    # Per-trial phase from filtered windows
    stim_phases = []
    if stim_windows:
        centre_idx = len(stim_windows[0]) // 2
        half_cycle = int(ANALYSIS_RATE / 2)
        lo_w = max(0, centre_idx - half_cycle)
        hi_w = min(len(stim_windows[0]), centre_idx + half_cycle)
        for w in stim_windows:
            peak_idx = lo_w + np.argmax(w[lo_w:hi_w])
            dt = (peak_idx - centre_idx) / ANALYSIS_RATE
            stim_phases.append((-dt * 2 * pi) % (2 * pi))
        stim_phases = np.array(stim_phases)

    # Blocked stim count
    blocked = [s for s in stims_without
               if not any(abs(s2.timestamp - s.timestamp) < 0.1 for s2 in stims_with)]
    n_fired = len(stims_with)
    n_blocked = len(blocked)

    duration_s = len(sig_ds) / ANALYSIS_RATE

    # ── Figure ────────────────────────────────────────────────────────
    fig = plt.figure(figsize=(18, 5))
    fig.suptitle(filename, fontsize=12, fontweight='bold')

    # (a) Stim-triggered average
    ax_a = fig.add_subplot(1, 3, 1)
    if stim_windows:
        for w in stim_windows:
            ax_a.plot(t_win, w, alpha=0.3, lw=0.5)
        avg = np.mean(stim_windows, axis=0)
        ax_a.plot(t_win, avg, 'k-', lw=2, label='mean')
    ax_a.axvline(0, color='red', ls='--', alpha=0.6, lw=1, label='stim')
    ax_a.set_xlabel('Time from stim (s)')
    ax_a.set_ylabel('Amplitude (SO band, \u00b5V)')
    ax_a.set_title(f'Stim-triggered average (n={len(stim_windows)})')
    ax_a.legend(fontsize=8)

    # (b) Stim phase
    ax_b = fig.add_subplot(1, 3, 2, projection='polar')
    if len(stim_phases) > 0:
        n_bins = 24
        counts, edges = np.histogram(stim_phases, bins=n_bins, range=(0, 2*pi))
        centres = (edges[:-1] + edges[1:]) / 2
        width = 2 * pi / n_bins
        ax_b.bar(centres, counts, width=width, alpha=0.6, edgecolor='k', lw=0.5,
                 color='#E74C3C')

        mean_phase = np.angle(np.mean(np.exp(1j * stim_phases))) % (2 * pi)
        ax_b.axvline(STIM_PHASE, color='green', ls='--', lw=2)
        ax_b.axvline(mean_phase, color='darkred', ls='-', lw=2)

        ax_b.set_thetagrids([0, 90, 180, 270],
                             labels=['peak', 'falling', 'trough', 'rising'], fontsize=9)
        ax_b.set_yticklabels([])

        error_deg = abs(mean_phase - STIM_PHASE) * 180 / pi
        if error_deg > 180:
            error_deg = 360 - error_deg

        ax_b.set_title(f'Stim phase (n={len(stim_phases)})\n'
                       f'green=target  red=actual  error={error_deg:.0f}\u00b0',
                       fontsize=9, pad=15)
    else:
        ax_b.set_title('No stims')

    # (c) Fired vs blocked
    ax_c = fig.add_subplot(1, 3, 3)
    labels = ['Fired', 'Blocked\n(IED)']
    values = [n_fired, n_blocked]
    colors = ['#2196F3', '#FF9800']
    bars = ax_c.bar(labels, values, color=colors, edgecolor='k', lw=0.5)
    for bar, val in zip(bars, values):
        if val > 0:
            ax_c.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                      str(val), ha='center', va='bottom', fontweight='bold')
    ax_c.set_ylabel('Count')
    rate_str = f'{(n_fired + n_blocked) / (duration_s / 60):.1f} det/min' if duration_s > 0 else ''
    ax_c.set_title(f'{duration_s:.0f}s recording \u2014 {rate_str}')
    ax_c.set_ylim(0, max(values + [1]) * 1.3)

    plt.tight_layout()

    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight')

    plt.show()
    return fig


print('Report function ready.')

---
## Run

In [ ]:
summary_rows = []

for npz_path in FILES:
    npz_path = Path(npz_path)
    name = npz_path.stem

    if not npz_path.exists():
        print(f'\n  \u2717 SKIPPING {npz_path.name} \u2014 file not found')
        continue

    hw_rate = get_file_info(npz_path)

    print(f'\n{"=" * 60}')
    print(f'  {npz_path.name}  ({hw_rate:.0f} Hz, ch={CHANNEL_ID})')
    print(f'{"=" * 60}')

    # Run WITH inhibition
    pipeline = make_pipeline(npz_path, hw_rate)
    events = pipeline.run_offline()

    detections = [e for e in events if e.event_type == EventType.SLOW_WAVE]
    stims = [e for e in events if e.event_type == EventType.STIM]
    print(f'  {len(detections)} detections, {len(stims)} stims (with inhibition)')

    # Run WITHOUT inhibition (for blocked count)
    stims_no = [e for e in make_pipeline_no_inh(npz_path, hw_rate).run_offline()
                if e.event_type == EventType.STIM]
    print(f'  {len(stims_no)} stims (without inhibition)')

    # Save detection log
    csv_path = OUTPUT_DIR / f'{name}_detections.csv'
    save_detection_log(csv_path, events, hw_rate)
    print(f'  Log: {csv_path.name}')

    # Report figure
    sig_ds = load_signal_ds(npz_path, hw_rate)
    fig_path = OUTPUT_DIR / f'{name}_report.png'
    make_report_figure(sig_ds, stims, stims_no, events, npz_path.name, save_path=fig_path)
    print(f'  Report: {fig_path.name}')

    # Summary
    duration_s = len(sig_ds) / ANALYSIS_RATE
    n_blocked = len(stims_no) - len(stims)
    summary_rows.append({
        'file': npz_path.name,
        'fs': f'{hw_rate:.0f}',
        'duration_s': f'{duration_s:.0f}',
        'detections': len(detections),
        'stims': len(stims),
        'blocked': n_blocked,
        'det_per_min': f'{len(detections) / (duration_s / 60):.1f}' if duration_s > 0 else '\u2014',
    })

print(f'\n{"=" * 60}')
print(f'  DONE \u2014 {len(summary_rows)} files')
print(f'  Output: {OUTPUT_DIR.resolve()}')
print(f'{"=" * 60}')

---
## Summary

In [ ]:
if summary_rows:
    header = list(summary_rows[0].keys())
    col_w = [max(len(h), max(len(str(r[h])) for r in summary_rows)) for h in header]

    print('  '.join(h.ljust(w) for h, w in zip(header, col_w)))
    print('  '.join('-' * w for w in col_w))
    for r in summary_rows:
        print('  '.join(str(r[h]).ljust(w) for h, w in zip(header, col_w)))

    summary_csv = OUTPUT_DIR / 'batch_summary.csv'
    with open(summary_csv, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=header)
        writer.writeheader()
        writer.writerows(summary_rows)
    print(f'\nSaved: {summary_csv}')
else:
    print('No files processed.')

---
## Detection log format

Each `_detections.csv` has columns:

| Column | Description |
| --- | --- |
| `event_type` | `SLOW_WAVE` or `STIM` |
| `timestamp_s` | Time in seconds from recording start |
| `sample_index` | Sample index at the **original hardware rate** |
| `frequency_hz` | Dominant SO frequency at detection |
| `amplitude` | Wavelet amplitude (SLOW_WAVE only) |
| `z_score` | Amplitude z-score (SLOW_WAVE only) |
| `pulse_index` | 1-indexed pulse number (STIM only) |
| `detection_time_s` | Timestamp of the triggering SLOW_WAVE (STIM only) |

Use `sample_index` to jump directly to the detection in the raw recording.

---
## Individual event inspection

Plot every detection with its surrounding context. Raw signal in grey,
SO-band filtered in blue. Blue line = detection, red dashed = stim(s).

Title shows the hardware sample index for lookup in the raw recording.
Set `MAX_EVENTS` to limit how many plots are drawn.

In [ ]:
# ── Plot individual events ────────────────────────────────────────────
# Pick one file to inspect every detection + stim in context.

INSPECT_FILE = FILES[0]      # ← change this to inspect a different file
CONTEXT_S = 4.0              # seconds of context either side
MAX_EVENTS = 50              # cap to avoid 500 plots

# Load and filter
npz_path = Path(INSPECT_FILE)
hw_rate = get_file_info(npz_path)
sig_ds = load_signal_ds(npz_path, hw_rate)
t_full = np.arange(len(sig_ds)) / ANALYSIS_RATE

nyq = ANALYSIS_RATE / 2.0
sos = butter(4, [FREQ_MIN / nyq, FREQ_MAX / nyq], btype='band', output='sos')
sig_so = sosfilt(sos, sig_ds)

# Run pipeline
events = make_pipeline(npz_path, hw_rate).run_offline()
detections = [e for e in events if e.event_type == EventType.SLOW_WAVE]
stims = [e for e in events if e.event_type == EventType.STIM]

# Index stims by detection time for easy lookup
from collections import defaultdict
stim_by_det = defaultdict(list)
for s in stims:
    stim_by_det[s.metadata.get('detection_time', -1)].append(s)

ctx_samples = int(CONTEXT_S * ANALYSIS_RATE)

n_plot = min(len(detections), MAX_EVENTS)
print(f'Plotting {n_plot} of {len(detections)} detections from {npz_path.name}')

for i, det in enumerate(detections[:n_plot]):
    t_det = det.timestamp
    centre = int(t_det * ANALYSIS_RATE)
    lo = max(0, centre - ctx_samples)
    hi = min(len(sig_so), centre + ctx_samples)

    t_win = t_full[lo:hi]
    sig_win = sig_so[lo:hi]
    sig_raw_win = sig_ds[lo:hi]

    fig, ax = plt.subplots(figsize=(14, 3))

    # Raw signal faint, filtered bold
    ax.plot(t_win, sig_raw_win, color='lightgrey', lw=0.5, label='raw' if i == 0 else '')
    ax.plot(t_win, sig_win, color='steelblue', lw=1, label='SO band' if i == 0 else '')

    # Detection
    ax.axvline(t_det, color='blue', ls='-', lw=1.5, alpha=0.7, label='detection')

    # Associated stims
    my_stims = stim_by_det.get(t_det, [])
    for j, s in enumerate(sorted(my_stims, key=lambda e: e.timestamp)):
        ax.axvline(s.timestamp, color='red', ls='--', lw=1.5, alpha=0.7,
                   label=f'stim {s.metadata.get("pulse_index", "")}' if j == 0 else '')

    freq = det.metadata.get('frequency', 0)
    amp = det.metadata.get('amplitude', 0)
    phase = det.metadata.get('detection_phase', 0)
    hw_idx = int(round(t_det * hw_rate))

    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Amplitude (\u00b5V)')
    ax.set_title(f'#{i+1}  t={t_det:.2f}s  sample={hw_idx}  '
                 f'f={freq:.2f}Hz  amp={amp:.0f}  '
                 f'phase={phase:.2f}rad  stims={len(my_stims)}',
                 fontsize=10)
    ax.legend(loc='upper right', fontsize=8)
    plt.tight_layout()
    plt.show()